# 10 — FD004: Full Model Ladder on the Hardest C-MAPSS Subset

FD004 combines both complexity axes Phase 12's notebooks tested in
isolation: 6 operating regimes (like FD002) AND 2 fault modes (like
FD003), on its own distinct set of 249 training / 248 test engines. This
is the subset that actually earns the word "generalization" — FD002 and
FD003 (`notebooks/09_fd002_fd003_generalization_check.ipynb`) established
the pipeline handles each axis separately; this notebook checks both at
once, with the same full model ladder and evaluation rigor as the
original FD001 build (Phases 1-6), not the lighter RSF-only check used
for the two intermediate subsets.

**A real bug caught building this notebook, worth stating up front:**
`data_loading.py`'s reference-statistics validation (added in Phase 1
specifically to catch a stale/corrupted mirror) caught that FD004's
train/test engine counts were transposed in this project's own reference
table (248/249 instead of the correct 249/248) — not a mirror problem,
a transcription error in this codebase from when FD002-004 support was
first wired up. Fixed in `data_loading.py`. Left in this notebook's
introduction rather than a silent fix, since it's a real demonstration of
why that validation step exists.


In [1]:
import sys
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import WeibullAFTFitter
from lifelines.utils import concordance_index
from sklearn.preprocessing import StandardScaler

sys.path.append(str(Path("..") / "src"))
from turbofan_rul import data_loading, evaluation, features, labeling, models, plotting

plotting.style()
SUBSET, CAP, DURATION_EPS = "FD004", 125, 1e-3
DATA_DIR = Path("..") / "data" / "raw"
PROCESSED_DIR = Path("..") / "data" / "processed"
MODELS_DIR = Path("..") / "models"
SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load, cluster, normalize, select sensors

In [2]:
data_loading.download_cmapss(SUBSET, data_dir=DATA_DIR)
fd004 = data_loading.load_subset(SUBSET, data_dir=DATA_DIR)
print(f"train: {fd004.train.shape} ({fd004.train.unit_number.nunique()} engines)")
print(f"test:  {fd004.test.shape} ({fd004.test.unit_number.nunique()} engines)")

kmeans_fd004 = features.fit_regime_clusters(fd004.train, n_clusters=6)
train_regime_labels = features.assign_regime_clusters(fd004.train, kmeans_fd004)
print(f"cluster sizes: {np.bincount(train_regime_labels)}")

norm_stats_fd004 = features.fit_regime_normalization_stats(fd004.train, SENSOR_COLS, train_regime_labels)
fd004_train_norm = features.apply_regime_normalization(fd004.train, SENSOR_COLS, train_regime_labels, norm_stats_fd004)

# Caveat: after within-regime z-scoring, any sensor that varies at all within
# its regime has std ~1, so this threshold only drops sensors that are
# constant within every regime -- it doesn't test informativeness.
std_fd004 = fd004_train_norm[SENSOR_COLS].std().sort_values()
STD_THRESHOLD = 0.05
kept_sensors_fd004 = std_fd004[std_fd004 > STD_THRESHOLD].index.tolist()
print(f"\n{len(kept_sensors_fd004)} kept sensors (dropped: {std_fd004[std_fd004 <= STD_THRESHOLD].index.tolist()})")

train: (61249, 26) (249 engines)
test:  (41214, 26) (248 engines)


cluster sizes: [15395  9224  9139  9091  9238  9162]

17 kept sensors (dropped: ['sensor_19', 'sensor_18', 'sensor_5', 'sensor_1'])


## 2. Labels, features, engine-level split

In [3]:
train_survival_fd004 = labeling.build_train_survival_frame(fd004_train_norm, cap=CAP)
t0 = time.time()
train_survival_fd004 = features.build_feature_matrix(train_survival_fd004, kept_sensors=kept_sensors_fd004)
print(f"feature build: {time.time() - t0:.1f}s, shape {train_survival_fd004.shape}")

fold_train_df, fold_val_df = models.train_val_split_by_engine(train_survival_fd004, val_fraction=0.2, random_state=42)

curated_cols_fd004 = ["cycle_count"]
for s in kept_sensors_fd004:
    curated_cols_fd004 += [f"{s}_roll15_mean", f"{s}_roll15_slope"]
META_COLS = {"unit_number", "time_cycles", "RUL", "duration", "event"}
full_cols_fd004 = [c for c in train_survival_fd004.columns if c not in META_COLS]
print(f"{len(curated_cols_fd004)} curated cols (Cox/Weibull), {len(full_cols_fd004)} full cols (RSF)")

feature build: 69.0s, shape (61249, 319)
35 curated cols (Cox/Weibull), 314 full cols (RSF)


## 3. Fit Cox PH, Weibull AFT, RSF -- same recipe as FD001, new data

All three models are fit on the **same 80% of training engines**
(`fold_train_df`), so the comparison below is like-for-like. (An earlier
version fit RSF on all 249 training engines while Cox and Weibull got the
80% split, which gave RSF roughly 25% more training engines and confounded
the comparison.)

In [4]:
train_scaled, val_scaled, curated_scaler = models.standardize_features(fold_train_df, fold_val_df, curated_cols_fd004)

t0 = time.time()
cox_fd004 = models.fit_cox_ph(train_scaled, curated_cols_fd004, penalizer=0.1)
print(f"Cox PH fit: {time.time() - t0:.1f}s")

t0 = time.time()
weibull_fit_df = train_scaled.copy()
weibull_fit_df["duration"] = weibull_fit_df["duration"] + DURATION_EPS
weibull_fd004 = WeibullAFTFitter(penalizer=0.1)
weibull_fd004.fit(weibull_fit_df[curated_cols_fd004 + ["duration", "event"]], duration_col="duration", event_col="event")
print(f"Weibull AFT fit: {time.time() - t0:.1f}s")

t0 = time.time()
rsf_fd004 = models.fit_random_survival_forest(
    fold_train_df, full_cols_fd004, row_stride=15, duration_eps=DURATION_EPS, random_state=42
)
print(f"RSF fit: {time.time() - t0:.1f}s (row_stride=15, coarser than FD001's 3 -- FD004 is the largest subset)")
print(f"Training engines -- Cox/Weibull: {train_scaled.unit_number.nunique()}, RSF: {fold_train_df.unit_number.nunique()}")

Cox PH fit: 0.7s


Weibull AFT fit: 3.0s


RSF fit: 16.7s (row_stride=15, coarser than FD001's 3 -- FD004 is the largest subset)
Training engines -- Cox/Weibull: 199, RSF: 199


## 4. Evaluate all three against the official FD004 test set

In [5]:
test_regime_labels = features.assign_regime_clusters(fd004.test, kmeans_fd004)
fd004_test_norm = features.apply_regime_normalization(fd004.test, SENSOR_COLS, test_regime_labels, norm_stats_fd004)
test_featured_fd004 = features.build_feature_matrix(fd004_test_norm, kept_sensors=kept_sensors_fd004)
test_survival_fd004 = labeling.build_test_survival_frame(test_featured_fd004, fd004.rul)
test_scaled_fd004 = test_survival_fd004.copy()
test_scaled_fd004[curated_cols_fd004] = curated_scaler.transform(test_survival_fd004[curated_cols_fd004])
true_rul_fd004 = test_survival_fd004["true_rul"].values

cox_pred = cox_fd004.predict_expectation(test_scaled_fd004[curated_cols_fd004]).values
weibull_pred = weibull_fd004.predict_percentile(test_scaled_fd004[curated_cols_fd004], p=0.5).values
weibull_low = weibull_fd004.predict_percentile(test_scaled_fd004[curated_cols_fd004], p=0.95).values
weibull_high = weibull_fd004.predict_percentile(test_scaled_fd004[curated_cols_fd004], p=0.05).values
rsf_out = models.predict_rul_from_survival(rsf_fd004, test_survival_fd004[full_cols_fd004])

rows = []
for name, pred, lo, hi in [
    ("Cox PH", cox_pred, None, None),
    ("Weibull AFT", weibull_pred, weibull_low, weibull_high),
    ("RSF", rsf_out["pred_rul"].values, rsf_out["pred_ci_low"].values, rsf_out["pred_ci_high"].values),
]:
    m = evaluation.evaluate_rul_predictions(true_rul_fd004, pred, lo, hi, model_name=name)
    m["cindex"] = concordance_index(true_rul_fd004, pred, event_observed=np.ones(len(true_rul_fd004)))
    rows.append(m)

fd004_results = pd.DataFrame(rows).set_index("model")[
    ["nasa_score", "nasa_score_mean", "cindex", "late_prediction_rate", "interval_coverage", "rmse", "mae"]
].round(3)
display(fd004_results)
fd004_results.to_csv(PROCESSED_DIR / "model_comparison_fd004.csv")

,nasa_score,nasa_score_mean,cindex,late_prediction_rate,interval_coverage,rmse,mae
model,,,,,,,
Cox PH,13532.086,54.565,0.811,0.456,NaN,34.912,27.464
Weibull AFT,17310.395,69.800,0.812,0.440,0.798,35.644,27.482
RSF,6811.609,27.466,0.849,0.512,0.706,29.535,22.088


## 5. FD001 vs. FD004: does the ranking and the relative story hold?

FD001 numbers are Phase 6's official-test-set results, read from the saved
comparison table. Concordance on both sides is engine-level C on point
predictions (`test_cindex`), so the columns are directly comparable.

In [6]:
fd001 = pd.read_csv(PROCESSED_DIR / "model_comparison_fd001.csv", index_col="model")
fd001 = fd001.rename(index={"Random Survival Forest": "RSF"})
fd001_results = pd.DataFrame({
    "fd001_cindex": fd001["test_cindex"],
    "fd001_nasa_mean": fd001["nasa_score"] / 100,   # 100 official FD001 test engines
    "fd001_rmse": fd001["rmse"],
})

side_by_side = fd001_results.join(
    fd004_results[["cindex", "nasa_score_mean", "rmse"]].rename(
        columns={"cindex": "fd004_cindex", "nasa_score_mean": "fd004_nasa_mean", "rmse": "fd004_rmse"}
    )
)
display(side_by_side.round(3))

rsf_still_best = fd004_results["cindex"].idxmax() == "RSF"
print(f"\nRSF still has the best concordance on FD004: {rsf_still_best}")
print(f"RSF still has the best NASA score on FD004:  {fd004_results['nasa_score'].idxmin() == 'RSF'}")

cw = fd004_results.loc[["Cox PH", "Weibull AFT"]]
print(f"\nCox vs. Weibull on FD004 -- lower RMSE: {cw['rmse'].idxmin()}, lower NASA score: {cw['nasa_score'].idxmin()}")
print("(On FD001 the two metrics picked different models: Cox on RMSE, Weibull on NASA score.)")

,fd001_cindex,fd001_nasa_mean,fd001_rmse,fd004_cindex,fd004_nasa_mean,fd004_rmse
model,,,,,,
Cox PH,0.821,16.725,22.013,0.811,54.565,34.912
Weibull AFT,0.818,12.882,23.141,0.812,69.800,35.644
RSF,0.864,4.625,16.265,0.849,27.466,29.535



RSF still has the best concordance on FD004: True
RSF still has the best NASA score on FD004:  True

Cox vs. Weibull on FD004 -- lower RMSE: Cox PH, lower NASA score: Cox PH
(On FD001 the two metrics picked different models: Cox on RMSE, Weibull on NASA score.)


## Verdict

**The RSF-leads ranking holds on FD004** -- best concordance (0.849 vs.
0.811 / 0.812) and best NASA score (27.5 per engine vs. 54.6 / 69.8), with
all three models trained on the same 199 engines. That's consistent with
the original FD001 result and the 5-fold CV check (Phase 11), now on a
structurally harder, entirely different subset: three independent pieces of
evidence (single split, cross-validation, distribution shift) for the same
conclusion, not one result reported three ways.

**How much harder FD004 is depends on the metric.** Concordance drops only
modestly from FD001 (RSF 0.864 -> 0.849; Cox and Weibull ~0.82 -> ~0.81),
so ranking engines by remaining life mostly survives the extra regimes and
fault mode. Point accuracy drops much more: RSF's NASA score per engine goes
from 4.6 to 27.5 and its RMSE from 16.3 to 29.5. Some of that gap reflects
this notebook's coarser `row_stride=15` (vs. FD001's `3`), adopted for
runtime, not only the harder problem.

**One part of the FD001 story does *not* carry over:** on FD001, RMSE and
the NASA score disagreed about Cox vs. Weibull (Cox better on RMSE, Weibull
better on NASA score). On FD004, Cox is better on both. So the
RMSE-vs-NASA disagreement is real but subset-specific -- it's evidence that
the two metrics *can* rank models differently, not a general rule about
these two models.

**Not yet tuned, not yet the coverage-fixed version of RSF.** This is the
untuned model ladder, the same recipe as FD001's Phases 3-5, applied to new
data. Hyperparameter tuning and RSF's interval-coverage fix were
deliberately not folded in here, so this notebook's result is a clean "does
the existing recipe generalize" answer, not conflated with "how much better
can FD004 get with more tuning effort."